# 06 — + LayerNorm

Builds on **05 +ElasticNet**: replaces BatchNorm on hidden_2 with LayerNorm.
BN stays on hidden_1; LN normalizes across features (not batch) on hidden_2.

**Cumulative stack:** L2 + BN + LN + Dropout + L1

**What to look for:**
- Compare activations on hidden_1 (BN) vs hidden_2 (LN)
- LayerNorm normalizes per-sample, so behavior is identical in train/eval
- Embedding evolution may differ from pure-BN version

In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from scripts.viz_export import ExperimentTracker

torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__}")
device = torch.device("cpu")

/Users/morgancooper/miniconda3/envs/nnpo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.10.0


## 1. Load MNIST

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

data_root = os.path.join(PROJECT_ROOT, "data")
full_train_dataset = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train_dataset, [48000, 12000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=torch.Generator().manual_seed(0))
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 48000, Val: 12000, Test: 10000


## 2. Define Model

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.ln2 = nn.LayerNorm(64)
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.drop2(torch.relu(self.ln2(self.fc2(x))))
        x = self.fc3(x)
        return x

model = MLP().to(device)
print(model)

MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## 3. Pick Input Samples & Viz Samples

In [4]:
# 5 input samples: one per distinct digit (first 5 unique)
seen_labels = set()
five_images, five_labels = [], []
for img, label in full_train_dataset:
    if label not in seen_labels:
        five_images.append(img)
        five_labels.append(int(label))
        seen_labels.add(label)
    if len(seen_labels) == 5:
        break
five_images = torch.stack(five_images)
print(f"Input samples: labels={five_labels}, shape={five_images.shape}")

# 500 viz samples: 50 per class
viz_images, viz_labels = [], []
class_counts = {c: 0 for c in range(10)}
for img, label in full_train_dataset:
    label = int(label)
    if class_counts[label] < 50:
        viz_images.append(img)
        viz_labels.append(label)
        class_counts[label] += 1
    if all(v >= 50 for v in class_counts.values()):
        break
viz_images = torch.stack(viz_images)
print(f"Viz samples: {len(viz_labels)} total, shape={viz_images.shape}")

Input samples: labels=[5, 0, 4, 1, 9], shape=torch.Size([5, 784])
Viz samples: 500 total, shape=torch.Size([500, 784])


## 4. Create Tracker

In [5]:
tracker = ExperimentTracker(
    run_id="mnist_layernorm",
    model_name="MNIST +LayerNorm",
    description="MLP + L2 + BN(h1) + LN(h2) + Dropout + L1",
    hyperparameters={
        "lr": 0.001, "batch_size": 512, "epochs": 10,
        "weight_decay": 1e-4, "l1_lambda": 1e-4, "dropout": 0.3,
    },
    model=model,
)

tracker.track("input", size=784)
tracker.track("hidden_1", model.fc1, size=128)
tracker.track("hidden_2", model.fc2, size=64)
tracker.track("output", model.fc3, size=10)

tracker.set_input_samples(five_images, five_labels)
tracker.set_viz_samples(viz_images, viz_labels)
tracker.enable_gradient_capture()
tracker.enable_loss_landscape()

ExperimentTracker: will write to /Users/morgancooper/NeuralNetworkProjectionOperator/scripts/../experimentation/runs/mnist_layernorm_v1


## 5. Train

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
l1_lambda = 1e-4

CHECKPOINT_EVERY = 2
num_epochs = 10

def evaluate(loader):
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            total_loss += criterion(out, y).item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

step = 0
global_batch = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(batch_x)
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(output, batch_y) + l1_lambda * l1_norm
        loss.backward()
        tracker.capture_gradients()
        optimizer.step()
        running_loss += loss.item()
        global_batch += 1
        if global_batch % CHECKPOINT_EVERY == 0:
            val_loss, val_acc = evaluate(val_loader)
            _, test_acc = evaluate(test_loader)
            tracker.compute_loss_landscape(batch_x, batch_y, criterion)
            tracker.save_checkpoint(step=step, epoch=epoch, metrics={
                "train_loss": running_loss / (batch_idx + 1),
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "test_accuracy": test_acc,
            })
            step += 1
            model.train()

    print(f"Epoch {epoch}: loss={running_loss / len(train_loader):.4f}")

print()
print(f"Total checkpoints saved: {step}")

  step_000.json (epoch=0, loss=1.9381, acc=0.4307, size=5.0MB)
  step_001.json (epoch=0, loss=1.6600, acc=0.6538, size=5.0MB)
  step_002.json (epoch=0, loss=1.4863, acc=0.7473, size=5.0MB)
  step_003.json (epoch=0, loss=1.3629, acc=0.7896, size=5.0MB)
  step_004.json (epoch=0, loss=1.2632, acc=0.8113, size=5.0MB)
  step_005.json (epoch=0, loss=1.1761, acc=0.8267, size=5.0MB)
  step_006.json (epoch=0, loss=1.1000, acc=0.8371, size=5.0MB)
  step_007.json (epoch=0, loss=1.0312, acc=0.8456, size=5.0MB)
  step_008.json (epoch=0, loss=0.9708, acc=0.8542, size=5.0MB)
  step_009.json (epoch=0, loss=0.9158, acc=0.8611, size=5.0MB)
  step_010.json (epoch=0, loss=0.8652, acc=0.8681, size=5.0MB)
  step_011.json (epoch=0, loss=0.8190, acc=0.8741, size=5.0MB)
  step_012.json (epoch=0, loss=0.7764, acc=0.8786, size=5.0MB)
  step_013.json (epoch=0, loss=0.7371, acc=0.8822, size=5.0MB)
  step_014.json (epoch=0, loss=0.7016, acc=0.8842, size=5.0MB)
  step_015.json (epoch=0, loss=0.6703, acc=0.8860, size

KeyboardInterrupt: 

## 6. Finalize

In [ ]:
tracker.finalize()
print(f"Run directory: {tracker.run_dir}")
print(f"Run ID: {tracker.run_id}")